# Java PathFinder (JPF) — Raw Output Extraction (Java)

This notebook performs **end-to-end execution of Java PathFinder (JPF)** against a Java repository and captures the **complete raw tool output** without modifying or filtering the results.

**Default repository:** [visvantha-testable/java-tool-testing-jacoco](https://github.com/visvantha-testable/java-tool-testing-jacoco)

> **Important:** Metrics are parsed only from explicit JPF console output. No Path Coverage values are inferred beyond what JPF emits.


## Section 1 — Install Dependencies

Install Java, Maven, Python packages, and build Java PathFinder (`jpf-core`, `java-17` branch).


In [ ]:
import platform
import shutil
import subprocess
import sys
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules
IS_LINUX = platform.system() == 'Linux'

if IS_COLAB or IS_LINUX:
    !apt-get update -qq
    !apt-get install -y openjdk-17-jdk maven git

!pip install -q pandas gitpython notebook jupyter

print('Java:')
subprocess.run(['java', '-version'], check=False)
if shutil.which('mvn'):
    print('\nMaven:')
    subprocess.run(['mvn', '-version'], check=False)
if shutil.which('gradle'):
    print('\nGradle:')
    subprocess.run(['gradle', '-version'], check=False)


## Section 2 — Configuration


In [ ]:
USE_GIT_REPO = True

REPO_URL = 'https://github.com/visvantha-testable/java-tool-testing-jacoco.git'

LOCAL_REPO = './workspace/java-tool-testing-jacoco'

WORKSPACE_DIR = './workspace'
OUTPUT_DIR = './outputs'
IF_CLONE_EXISTS = 'reuse'
CLONE_DEPTH = 1
RAW_OUTPUT_PREVIEW_LINES = 150
JPF_REPO_URL = 'https://github.com/javapathfinder/jpf-core.git'
JPF_BRANCH = 'java-17'

# Local mode example:
# USE_GIT_REPO = False
# LOCAL_REPO = './workspace/java-tool-testing-jacoco'

# Colab example:
# USE_GIT_REPO = False
# LOCAL_REPO = '/content/java-tool-testing-jacoco'


## Section 3 — Imports and Utility Functions


In [ ]:
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import display

JACOCO_UTILS_ROOT = Path('..') / 'JaCoCo Coverage' / 'tool'
sys.path.insert(0, str(JACOCO_UTILS_ROOT.resolve()))
sys.path.insert(0, str(Path('tool').resolve()))

from __future__ import annotations

import csv
import os
import platform
import re
import shutil
import subprocess
import sys
import time
import xml.etree.ElementTree as ET
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from git import Repo
from git.exc import GitCommandError, InvalidGitRepositoryError

EXCLUDED_DIR_NAMES = {
    ".git", "target", "build", "bin", ".idea", ".gradle", ".mvn", "out", "node_modules",
}
COUNTER_TYPES = ["INSTRUCTION", "BRANCH", "LINE", "METHOD", "CLASS", "COMPLEXITY"]
PY = sys.executable
PACKAGE_RE = re.compile(r"^\s*package\s+([\w.]+)\s*;", re.MULTILINE)


@dataclass
class CommandResult:
    command: list[str]
    stdout: str
    stderr: str
    exit_code: int
    execution_time_seconds: float


@dataclass
class BuildStatus:
    build_tool: str
    build_command: list[str]
    jacoco_command: list[str]
    build_result: CommandResult | None = None
    jacoco_result: CommandResult | None = None
    build_success: bool = False
    test_success: bool = False
    report_generated: bool = False
    report_dir: Path | None = None
    jacoco_exec: Path | None = None
    jacoco_xml: Path | None = None
    jacoco_csv: Path | None = None
    index_html: Path | None = None


class NotebookLogger:
    def __init__(self, error_log_path: Path) -> None:
        self.error_log_path = error_log_path
        self.error_log_path.parent.mkdir(parents=True, exist_ok=True)
        self._errors: list[dict[str, str]] = []
        self.write_errors()

    def info(self, message: str) -> None:
        timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        print(f"[{timestamp}] INFO: {message}")

    def error(self, message: str, step: str = "notebook") -> None:
        timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        line = f"[{timestamp}] ERROR: {message}\n"
        print(line, end="")
        self._errors.append({"timestamp": timestamp, "step": step, "error": message})
        self.write_errors()

    def write_errors(self) -> None:
        with self.error_log_path.open("w", encoding="utf-8", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=["timestamp", "step", "error"])
            writer.writeheader()
            writer.writerows(self._errors)


def ensure_output_dir(output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)


def project_runtimes_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    candidates: list[Path] = []
    for parent in [current, *current.parents]:
        candidate = parent / "runtimes"
        if candidate.is_dir():
            candidates.append(candidate)
    for candidate in candidates:
        if (candidate / "jdk-21").exists() or any(candidate.glob("apache-maven-*")):
            return candidate
    return candidates[0] if candidates else current / "runtimes"


def configure_java_runtime(logger: NotebookLogger) -> dict[str, str]:
    runtimes = project_runtimes_root()
    env = os.environ.copy()
    jdk_candidates = [
        runtimes / "jdk-21",
        Path(env.get("JAVA_HOME", "")),
        Path(r"C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot"),
    ]
    for candidate in jdk_candidates:
        java_bin = candidate / "bin"
        java_exe = java_bin / ("java.exe" if platform.system() == "Windows" else "java")
        if java_exe.exists():
            env["JAVA_HOME"] = str(candidate)
            env["PATH"] = str(java_bin) + os.pathsep + env.get("PATH", "")
            logger.info(f"Using JAVA_HOME={candidate}")
            break
    else:
        logger.info("Using system Java from PATH.")
    return env


def java_version_text(env: dict[str, str]) -> str:
    completed = subprocess.run(
        ["java", "-version"],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        check=False,
        env=env,
    )
    return combine_streams(completed.stdout, completed.stderr).strip()


def resolve_maven_command(repo_path: Path, logger: NotebookLogger) -> list[str]:
    if platform.system() == "Windows":
        wrapper = repo_path / "mvnw.cmd"
        if wrapper.exists():
            return [str(wrapper)]
    else:
        wrapper = repo_path / "mvnw"
        if wrapper.exists():
            return [str(wrapper)]
    runtimes = project_runtimes_root(repo_path)
    bundled = runtimes / "apache-maven-3.9.6" / "bin" / ("mvn.cmd" if platform.system() == "Windows" else "mvn")
    if bundled.exists():
        logger.info(f"Using bundled Maven: {bundled}")
        return [str(bundled)]
    return ["mvn"]


def resolve_gradle_command(repo_path: Path, logger: NotebookLogger) -> list[str]:
    if platform.system() == "Windows":
        wrapper = repo_path / "gradlew.bat"
    else:
        wrapper = repo_path / "gradlew"
    if wrapper.exists():
        if platform.system() != "Windows":
            wrapper.chmod(wrapper.stat().st_mode | 0o111)
        logger.info(f"Using Gradle wrapper: {wrapper}")
        return [str(wrapper)]
    return ["gradle"]


def derive_clone_path(repo_url: str, workspace_dir: Path) -> Path:
    repo_name = repo_url.rstrip("/").removesuffix(".git").split("/")[-1]
    if not repo_name:
        raise ValueError(f"Unable to derive repository name from URL: {repo_url}")
    return workspace_dir / repo_name


def validate_repo_url(repo_url: str) -> None:
    if not repo_url or not isinstance(repo_url, str):
        raise ValueError("REPO_URL must be a non-empty string.")
    if not (repo_url.startswith("https://") or repo_url.startswith("git@") or repo_url.startswith("http://")):
        raise ValueError(f"Invalid repository URL format: {repo_url}")


def clone_or_reuse_repository(
    repo_url: str,
    workspace_dir: Path,
    if_clone_exists: str,
    logger: NotebookLogger,
    clone_depth: int | None = None,
) -> Path:
    validate_repo_url(repo_url)
    workspace_dir.mkdir(parents=True, exist_ok=True)
    clone_path = derive_clone_path(repo_url, workspace_dir)
    if clone_path.exists():
        if if_clone_exists == "reclone":
            logger.info(f"Removing existing clone at {clone_path}")
            shutil.rmtree(clone_path)
        elif if_clone_exists == "reuse":
            logger.info(f"Reusing existing clone at {clone_path}")
            return clone_path.resolve()
        else:
            raise ValueError('IF_CLONE_EXISTS must be either "reuse" or "reclone"')
    logger.info(f"Cloning {repo_url} into {clone_path}")
    clone_kwargs: dict[str, Any] = {"depth": clone_depth} if clone_depth else {}
    try:
        Repo.clone_from(repo_url, clone_path, **clone_kwargs)
    except GitCommandError as exc:
        logger.error(f"Git clone failed: {exc}", step="clone")
        raise
    return clone_path.resolve()


def validate_local_repo_path(local_repo_path: Path, logger: NotebookLogger) -> Path:
    if not local_repo_path.exists():
        msg = f"Local repository path does not exist: {local_repo_path}"
        logger.error(msg, step="repository")
        raise FileNotFoundError(msg)
    if not local_repo_path.is_dir():
        msg = f"Local repository path is not a directory: {local_repo_path}"
        logger.error(msg, step="repository")
        raise NotADirectoryError(msg)
    build_files = [
        local_repo_path / "pom.xml",
        local_repo_path / "build.gradle",
        local_repo_path / "build.gradle.kts",
    ]
    if not any(path.exists() for path in build_files):
        msg = "No pom.xml, build.gradle, or build.gradle.kts found in repository."
        logger.error(msg, step="repository")
        raise FileNotFoundError(msg)
    return local_repo_path.resolve()


def resolve_repository_path(
    use_git_repo: bool,
    repo_url: str,
    local_repo: str | Path,
    workspace_dir: Path,
    if_clone_exists: str,
    logger: NotebookLogger,
    clone_depth: int | None = None,
) -> Path:
    if use_git_repo:
        return clone_or_reuse_repository(repo_url, workspace_dir, if_clone_exists, logger, clone_depth)
    return validate_local_repo_path(Path(local_repo), logger)


def detect_build_tool(repo_path: Path) -> str:
    if (repo_path / "pom.xml").exists():
        return "Maven"
    if (repo_path / "build.gradle.kts").exists():
        return "Gradle Kotlin DSL"
    if (repo_path / "build.gradle").exists():
        return "Gradle"
    raise FileNotFoundError("Unable to detect Maven or Gradle build files.")


def discover_java_files(repo_path: Path) -> list[Path]:
    files: list[Path] = []
    for path in repo_path.rglob("*.java"):
        if any(part in EXCLUDED_DIR_NAMES for part in path.parts):
            continue
        files.append(path.resolve())
    return sorted(files)


def extract_package_name(java_path: Path) -> str:
    try:
        text = java_path.read_text(encoding="utf-8")
    except (OSError, UnicodeDecodeError):
        return ""
    match = PACKAGE_RE.search(text)
    return match.group(1) if match else ""


def save_java_inventory(java_files: list[Path], output_csv: Path) -> None:
    rows = [
        {
            "file_path": str(path),
            "file_name": path.name,
            "package": extract_package_name(path),
            "directory": str(path.parent),
        }
        for path in java_files
    ]
    pd.DataFrame(rows).to_csv(output_csv, index=False)


def count_all_files(repo_path: Path) -> int:
    total = 0
    for path in repo_path.rglob("*"):
        if path.is_file() and not any(part in EXCLUDED_DIR_NAMES for part in path.parts):
            total += 1
    return total


def compute_repository_stats(
    repo_path: Path,
    java_files: list[Path],
    build_tool: str,
    java_version: str,
) -> dict[str, Any]:
    total_size = sum(path.stat().st_size for path in java_files)
    return {
        "repository_name": repo_path.name,
        "repository_location": str(repo_path),
        "build_tool": build_tool,
        "java_version": java_version.replace("\n", " | "),
        "total_java_files": len(java_files),
        "total_files": count_all_files(repo_path),
        "repository_size_bytes": total_size,
    }


def combine_streams(stdout: str, stderr: str) -> str:
    raw = stdout
    if stderr:
        if raw and not raw.endswith("\n"):
            raw += "\n"
        raw += stderr
    return raw


def run_shell_command(
    command: list[str],
    cwd: Path,
    env: dict[str, str],
    logger: NotebookLogger,
    step: str,
) -> CommandResult:
    logger.info(f"Executing ({step}): {' '.join(command)}")
    started = time.perf_counter()
    completed = subprocess.run(
        command,
        cwd=str(cwd),
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        check=False,
        env=env,
    )
    elapsed = round(time.perf_counter() - started, 5)
    result = CommandResult(
        command=command,
        stdout=completed.stdout,
        stderr=completed.stderr,
        exit_code=completed.returncode,
        execution_time_seconds=elapsed,
    )
    if completed.returncode != 0:
        logger.error(
            combine_streams(completed.stdout, completed.stderr) or f"Command failed with exit code {completed.returncode}",
            step=step,
        )
    return result


def find_jacoco_report_dirs(repo_path: Path) -> list[Path]:
    report_dirs: list[Path] = []
    for xml_path in repo_path.rglob("jacoco.xml"):
        if ".git" in xml_path.parts:
            continue
        parent = str(xml_path.parent).replace("\\", "/")
        if "/site/jacoco" in parent or "/reports/jacoco" in parent or parent.endswith("/jacoco"):
            report_dirs.append(xml_path.parent.resolve())
    return sorted(set(report_dirs))


def find_jacoco_exec_files(repo_path: Path) -> list[Path]:
    exec_files: list[Path] = []
    for exec_path in repo_path.rglob("jacoco.exec"):
        if ".git" in exec_path.parts:
            continue
        exec_files.append(exec_path.resolve())
    return sorted(exec_files)


def choose_primary_report_dir(report_dirs: list[Path]) -> Path | None:
    if not report_dirs:
        return None
    best_dir = report_dirs[0]
    best_score = -1
    for report_dir in report_dirs:
        xml_path = report_dir / "jacoco.xml"
        if not xml_path.exists():
            continue
        counters = parse_counter_map(xml_path)
        score = sum(counters.get(counter, {}).get("covered", 0) + counters.get(counter, {}).get("missed", 0) for counter in COUNTER_TYPES)
        if score > best_score:
            best_score = score
            best_dir = report_dir
    return best_dir


def choose_primary_exec(exec_files: list[Path], report_dir: Path | None) -> Path | None:
    if report_dir is not None:
        maven_exec = report_dir.parent.parent / "jacoco.exec"
        if maven_exec.exists():
            return maven_exec.resolve()
        gradle_exec = report_dir.parent / "jacoco.exec"
        if gradle_exec.exists():
            return gradle_exec.resolve()
    return exec_files[-1] if exec_files else None


def coverage_percent(covered: int, missed: int) -> float:
    total = covered + missed
    if total == 0:
        return 100.0
    return round(covered * 100.0 / total, 2)


def parse_counter_map(xml_path: Path) -> dict[str, dict[str, int]]:
    root = ET.parse(xml_path).getroot()
    counters: dict[str, dict[str, int]] = {}
    for counter in root.findall("counter"):
        counter_type = counter.get("type", "")
        counters[counter_type] = {
            "missed": int(counter.get("missed", "0")),
            "covered": int(counter.get("covered", "0")),
        }
    return counters


def counters_to_metrics_rows(counters: dict[str, dict[str, int]]) -> list[dict[str, Any]]:
    label_map = {
        "INSTRUCTION": "Instruction",
        "BRANCH": "Branch",
        "LINE": "Line",
        "METHOD": "Method",
        "CLASS": "Class",
        "COMPLEXITY": "Complexity",
    }
    rows: list[dict[str, Any]] = []
    for counter_type in COUNTER_TYPES:
        values = counters.get(counter_type, {"missed": 0, "covered": 0})
        covered = values.get("covered", 0)
        missed = values.get("missed", 0)
        rows.append(
            {
                "metric_name": f"{label_map[counter_type]} Covered",
                "covered": covered,
                "missed": missed,
                "coverage_percent": coverage_percent(covered, missed),
            }
        )
    return rows


def element_counters(element: ET.Element) -> dict[str, dict[str, int]]:
    counters: dict[str, dict[str, int]] = {}
    for counter in element.findall("counter"):
        counter_type = counter.get("type", "")
        counters[counter_type] = {
            "missed": int(counter.get("missed", "0")),
            "covered": int(counter.get("covered", "0")),
        }
    return counters


def build_package_summary_rows(xml_path: Path) -> list[dict[str, Any]]:
    root = ET.parse(xml_path).getroot()
    rows: list[dict[str, Any]] = []
    for package in root.findall("package"):
        package_name = package.get("name", "").replace("/", ".")
        counters = element_counters(package)
        rows.append(
            {
                "package": package_name,
                "instruction_coverage": coverage_percent(
                    counters.get("INSTRUCTION", {}).get("covered", 0),
                    counters.get("INSTRUCTION", {}).get("missed", 0),
                ),
                "branch_coverage": coverage_percent(
                    counters.get("BRANCH", {}).get("covered", 0),
                    counters.get("BRANCH", {}).get("missed", 0),
                ),
                "line_coverage": coverage_percent(
                    counters.get("LINE", {}).get("covered", 0),
                    counters.get("LINE", {}).get("missed", 0),
                ),
                "method_coverage": coverage_percent(
                    counters.get("METHOD", {}).get("covered", 0),
                    counters.get("METHOD", {}).get("missed", 0),
                ),
                "class_coverage": coverage_percent(
                    counters.get("CLASS", {}).get("covered", 0),
                    counters.get("CLASS", {}).get("missed", 0),
                ),
                "complexity_coverage": coverage_percent(
                    counters.get("COMPLEXITY", {}).get("covered", 0),
                    counters.get("COMPLEXITY", {}).get("missed", 0),
                ),
            }
        )
    return rows


def build_class_summary_rows(xml_path: Path) -> list[dict[str, Any]]:
    root = ET.parse(xml_path).getroot()
    rows: list[dict[str, Any]] = []
    for package in root.findall("package"):
        package_name = package.get("name", "").replace("/", ".")
        for class_el in package.findall("class"):
            class_name = class_el.get("name", "").split("/")[-1]
            counters = element_counters(class_el)
            rows.append(
                {
                    "package": package_name,
                    "class": class_name,
                    "instruction_coverage": coverage_percent(
                        counters.get("INSTRUCTION", {}).get("covered", 0),
                        counters.get("INSTRUCTION", {}).get("missed", 0),
                    ),
                    "branch_coverage": coverage_percent(
                        counters.get("BRANCH", {}).get("covered", 0),
                        counters.get("BRANCH", {}).get("missed", 0),
                    ),
                    "line_coverage": coverage_percent(
                        counters.get("LINE", {}).get("covered", 0),
                        counters.get("LINE", {}).get("missed", 0),
                    ),
                    "method_coverage": coverage_percent(
                        counters.get("METHOD", {}).get("covered", 0),
                        counters.get("METHOD", {}).get("missed", 0),
                    ),
                    "complexity_coverage": coverage_percent(
                        counters.get("COMPLEXITY", {}).get("covered", 0),
                        counters.get("COMPLEXITY", {}).get("missed", 0),
                    ),
                }
            )
    return rows


def build_repository_metrics_row(
    xml_path: Path,
    repo_stats: dict[str, Any],
    total_execution_time: float,
) -> dict[str, Any]:
    root = ET.parse(xml_path).getroot()
    counters = parse_counter_map(xml_path)
    packages = root.findall("package")
    classes = root.findall(".//class")
    return {
        "Total Packages": len(packages),
        "Total Classes": len(classes),
        "Instruction Coverage %": coverage_percent(
            counters.get("INSTRUCTION", {}).get("covered", 0),
            counters.get("INSTRUCTION", {}).get("missed", 0),
        ),
        "Branch Coverage %": coverage_percent(
            counters.get("BRANCH", {}).get("covered", 0),
            counters.get("BRANCH", {}).get("missed", 0),
        ),
        "Line Coverage %": coverage_percent(
            counters.get("LINE", {}).get("covered", 0),
            counters.get("LINE", {}).get("missed", 0),
        ),
        "Method Coverage %": coverage_percent(
            counters.get("METHOD", {}).get("covered", 0),
            counters.get("METHOD", {}).get("missed", 0),
        ),
        "Class Coverage %": coverage_percent(
            counters.get("CLASS", {}).get("covered", 0),
            counters.get("CLASS", {}).get("missed", 0),
        ),
        "Complexity Coverage %": coverage_percent(
            counters.get("COMPLEXITY", {}).get("covered", 0),
            counters.get("COMPLEXITY", {}).get("missed", 0),
        ),
        "Execution Time (seconds)": total_execution_time,
        "Repository Name": repo_stats["repository_name"],
        "Build Tool": repo_stats["build_tool"],
    }


def copy_artifact(source: Path | None, destination: Path) -> bool:
    if source is None or not source.exists():
        return False
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    return True


def execute_build_and_jacoco(
    repo_path: Path,
    build_tool: str,
    env: dict[str, str],
    logger: NotebookLogger,
) -> tuple[BuildStatus, str]:
    chunks: list[str] = []
    status = BuildStatus(build_tool=build_tool, build_command=[], jacoco_command=[])

    if build_tool == "Maven":
        maven = resolve_maven_command(repo_path, logger)
        status.build_command = [*maven, "clean", "test"]
        status.jacoco_command = [*maven, "clean", "test", "jacoco:report"]
    else:
        gradle = resolve_gradle_command(repo_path, logger)
        status.build_command = [*gradle, "clean", "test"]
        status.jacoco_command = [*gradle, "clean", "test", "jacocoTestReport"]

    build_result = run_shell_command(status.build_command, repo_path, env, logger, "build")
    status.build_result = build_result
    status.build_success = build_result.exit_code == 0
    status.test_success = "BUILD SUCCESS" in combine_streams(build_result.stdout, build_result.stderr) or build_result.exit_code == 0
    chunks.append(f"===== {' '.join(status.build_command)} =====")
    chunks.append(combine_streams(build_result.stdout, build_result.stderr))

    report_dirs = find_jacoco_report_dirs(repo_path)
    if not report_dirs and build_tool == "Maven":
        for module_pom in repo_path.rglob("pom.xml"):
            module_dir = module_pom.parent
            if module_dir == repo_path:
                continue
            module_text = module_pom.read_text(encoding="utf-8", errors="replace")
            if "jacoco-maven-plugin" not in module_text:
                continue
            jacoco_result = run_shell_command(
                [*resolve_maven_command(repo_path, logger), "jacoco:report"],
                module_dir,
                env,
                logger,
                "jacoco",
            )
            status.jacoco_result = jacoco_result
            chunks.append(f"===== module jacoco report {module_dir} =====")
            chunks.append(combine_streams(jacoco_result.stdout, jacoco_result.stderr))
        report_dirs = find_jacoco_report_dirs(repo_path)
        status.test_success = status.test_success or (status.jacoco_result.exit_code == 0 if status.jacoco_result else False)

    exec_files = find_jacoco_exec_files(repo_path)
    status.report_dir = choose_primary_report_dir(report_dirs)
    status.jacoco_exec = choose_primary_exec(exec_files, status.report_dir)
    if status.report_dir is not None:
        status.jacoco_xml = status.report_dir / "jacoco.xml"
        status.jacoco_csv = status.report_dir / "jacoco.csv"
        status.index_html = status.report_dir / "index.html"
        status.report_generated = status.jacoco_xml.exists()
    if not status.report_generated:
        logger.error("JaCoCo report files were not found after build.", step="jacoco")

    return status, "\n".join(chunks)


def collect_outputs(
    status: BuildStatus,
    repo_stats: dict[str, Any],
    output_dir: Path,
    total_execution_time: float,
    logger: NotebookLogger,
) -> dict[str, Any]:
    ensure_output_dir(output_dir)
    copied = {
        "jacoco.exec": copy_artifact(status.jacoco_exec, output_dir / "jacoco.exec"),
        "jacoco.xml": copy_artifact(status.jacoco_xml, output_dir / "jacoco.xml"),
        "jacoco.csv": copy_artifact(status.jacoco_csv, output_dir / "jacoco.csv"),
        "index.html": copy_artifact(status.index_html, output_dir / "index.html"),
    }
    if not copied["jacoco.xml"]:
        raise FileNotFoundError("Unable to copy jacoco.xml to outputs.")

    xml_path = output_dir / "jacoco.xml"
    metrics_df = pd.DataFrame(counters_to_metrics_rows(parse_counter_map(xml_path)))
    metrics_df.to_csv(output_dir / "jacoco_metrics.csv", index=False)

    package_df = pd.DataFrame(build_package_summary_rows(xml_path))
    package_df.to_csv(output_dir / "package_summary.csv", index=False)

    class_df = pd.DataFrame(build_class_summary_rows(xml_path))
    class_df.to_csv(output_dir / "class_summary.csv", index=False)

    repository_metrics = build_repository_metrics_row(xml_path, repo_stats, total_execution_time)
    pd.DataFrame([repository_metrics]).to_csv(output_dir / "repository_metrics.csv", index=False)

    return {
        "copied": copied,
        "metrics_df": metrics_df,
        "package_df": package_df,
        "class_df": class_df,
        "repository_metrics": repository_metrics,
    }


def preview_raw_output(raw_text: str, preview_lines: int, output_path: Path) -> None:
    lines = raw_text.splitlines()
    print(f"\n{'=' * 80}")
    print(f"RAW JACOCO CONSOLE OUTPUT PREVIEW (first {preview_lines} lines)")
    print(f"{'=' * 80}\n")
    if not lines:
        print("(empty raw output)")
        return
    print("\n".join(lines[:preview_lines]))
    remaining = len(lines) - preview_lines
    if remaining > 0:
        print(f"\n... ({remaining} more lines saved to {output_path})")


"""Java PathFinder (JPF) raw output extraction helpers."""
from __future__ import annotations

import csv
import os
import platform
import re
import shutil
import subprocess
import sys
import time
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd

TOOL_ROOT = Path(__file__).resolve().parent
JACOCO_TOOL_ROOT = TOOL_ROOT.parent.parent / "JaCoCo Coverage" / "tool"
if str(JACOCO_TOOL_ROOT) not in sys.path:
    sys.path.insert(0, str(JACOCO_TOOL_ROOT))

from _jacoco_notebook_utils import (  # noqa: E402
    EXCLUDED_DIR_NAMES,
    CommandResult,
    combine_streams,
    configure_java_runtime,
    detect_build_tool,
    discover_java_files,
    ensure_output_dir,
    extract_package_name,
    project_runtimes_root,
    resolve_gradle_command,
    resolve_maven_command,
    resolve_repository_path,
    run_shell_command,
)

MAIN_METHOD_RE = re.compile(
    r"public\s+static\s+void\s+main\s*\(\s*String(?:\s*\[\s*\])?\s+\w+\s*\)",
    re.MULTILINE,
)
CLASS_NAME_RE = re.compile(r"(?:public\s+)?(?:final\s+)?class\s+(\w+)")
STATISTICS_BLOCK_RE = re.compile(
    r"={50,}\s*statistics\s*(.*?)\s*={50,}\s*search finished",
    re.IGNORECASE | re.DOTALL,
)
STATES_RE = re.compile(
    r"states:\s*new=(\d+),visited=(\d+),backtracked=(\d+),end=(\d+)",
    re.IGNORECASE,
)
SEARCH_DEPTH_RE = re.compile(r"search:\s*maxDepth=(\d+)", re.IGNORECASE)
CHOICE_GENERATORS_RE = re.compile(r"choice generators:\s*(.+)", re.IGNORECASE)
TRANSITION_COUNT_RE = re.compile(r"transition #(\d+)", re.IGNORECASE)
ERROR_STATE_RE = re.compile(r"error #(\d+):", re.IGNORECASE)
EXCEPTION_RE = re.compile(
    r"(?:gov\.nasa\.jpf\.vm\.NoUncaughtExceptionsProperty|java\.lang\.\w+Exception|java\.lang\.\w+Error)",
    re.IGNORECASE,
)
DEADLOCK_RE = re.compile(r"deadlock", re.IGNORECASE)
TRACE_SECTION_RE = re.compile(r"(={50,}\s*trace #\d+.*?)(?=={50,}|\Z)", re.DOTALL | re.IGNORECASE)

PATH_METRICS = [
    "Path Execution Tracking",
    "Complete Coverage Path Verification",
    "Partial Path Coverage Detection",
    "Nested Condition Path Testing",
    "Loop Path Detection",
    "Unreachable Path Detection",
    "Exception Path Handling",
    "Multi-Function Path Tracking",
    "Path Detection Testing",
]

METRIC_EVIDENCE_PATTERNS: dict[str, list[tuple[str, re.Pattern[str]]]] = {
    "Path Execution Tracking": [
        ("transition", re.compile(r"transition #\d+", re.IGNORECASE)),
        ("trace", re.compile(r"trace #\d+", re.IGNORECASE)),
        ("instructions", re.compile(r"instructions:\s*\d+", re.IGNORECASE)),
    ],
    "Complete Coverage Path Verification": [
        ("states end", re.compile(r"states:.*end=\d+", re.IGNORECASE)),
        ("search finished", re.compile(r"search finished", re.IGNORECASE)),
    ],
    "Partial Path Coverage Detection": [
        ("visited states", re.compile(r"states:.*visited=\d+", re.IGNORECASE)),
        ("backtracked", re.compile(r"backtracked=\d+", re.IGNORECASE)),
    ],
    "Nested Condition Path Testing": [
        ("choice generators", re.compile(r"choice generators:", re.IGNORECASE)),
        ("constraints", re.compile(r"constraints=\d+", re.IGNORECASE)),
    ],
    "Loop Path Detection": [
        ("backtracked", re.compile(r"backtracked=\d+", re.IGNORECASE)),
        ("maxDepth", re.compile(r"maxDepth=\d+", re.IGNORECASE)),
    ],
    "Unreachable Path Detection": [
        ("end states", re.compile(r"states:.*end=\d+", re.IGNORECASE)),
        ("backtracked", re.compile(r"backtracked=\d+", re.IGNORECASE)),
    ],
    "Exception Path Handling": [
        ("NoUncaughtExceptionsProperty", re.compile(r"NoUncaughtExceptionsProperty", re.IGNORECASE)),
        ("Exception trace", re.compile(r"java\.lang\.\w+Exception", re.IGNORECASE)),
    ],
    "Multi-Function Path Tracking": [
        ("transition stack", re.compile(r"at\s+[\w.$]+\([\w/$.]+:\d+\)", re.IGNORECASE)),
        ("loaded methods", re.compile(r"loaded code:\s*classes=\d+,methods=\d+", re.IGNORECASE)),
    ],
    "Path Detection Testing": [
        ("visited states", re.compile(r"visited=\d+", re.IGNORECASE)),
        ("new states", re.compile(r"new=\d+", re.IGNORECASE)),
        ("statistics", re.compile(r"statistics", re.IGNORECASE)),
    ],
}

JPF_OPTIONAL_ARTIFACTS = [
    "search_graph.txt",
    "error_trace.txt",
    "visited_states.txt",
    "path_report.txt",
]


@dataclass
class JavaClassInfo:
    file_path: Path
    package: str
    class_name: str
    qualified_name: str
    has_main: bool


@dataclass
class JpfInstallStatus:
    jpf_home: Path
    run_jpf_jar: Path
    site_properties: Path
    install_command: list[str]
    build_command: list[str]
    install_result: CommandResult | None = None
    build_result: CommandResult | None = None
    install_success: bool = False
    build_success: bool = False


@dataclass
class JpfClassRun:
    qualified_name: str
    jpf_file: Path
    command: list[str]
    stdout: str
    stderr: str
    exit_code: int
    execution_time_seconds: float
    success: bool
    metrics: list[dict[str, str]] = field(default_factory=list)


class JpfNotebookLogger:
    def __init__(self, error_log_path: Path) -> None:
        self.error_log_path = error_log_path
        self.error_log_path.parent.mkdir(parents=True, exist_ok=True)
        self._errors: list[dict[str, str]] = []
        self.write_errors()

    def info(self, message: str) -> None:
        timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        print(f"[{timestamp}] INFO: {message}")

    def error(self, message: str, step: str = "notebook", class_name: str = "") -> None:
        timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        line = f"[{timestamp}] ERROR: {message}\n"
        print(line, end="")
        self._errors.append(
            {
                "timestamp": timestamp,
                "step": step,
                "class": class_name,
                "error": message,
            }
        )
        self.write_errors()

    def write_errors(self) -> None:
        with self.error_log_path.open("w", encoding="utf-8", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=["timestamp", "step", "class", "error"])
            writer.writeheader()
            writer.writerows(self._errors)


def jpf_path(value: Path | str) -> str:
    return str(value).replace("\\", "/")


def resolve_jpf_home(logger: JpfNotebookLogger, workspace_dir: Path) -> Path:
    candidates = [
        project_runtimes_root() / "jpf-core",
        workspace_dir / "jpf-core",
        Path.home() / ".jpf" / "jpf-core",
    ]
    for candidate in candidates:
        if (candidate / "build" / "RunJPF.jar").exists():
            logger.info(f"Using existing JPF installation: {candidate}")
            return candidate.resolve()
    return candidates[0].resolve()


def resolve_gradle_wrapper(jpf_home: Path) -> list[str]:
    if platform.system() == "Windows":
        wrapper = jpf_home / "gradlew.bat"
    else:
        wrapper = jpf_home / "gradlew"
    if wrapper.exists():
        if platform.system() != "Windows":
            wrapper.chmod(wrapper.stat().st_mode | 0o111)
        return [str(wrapper)]
    return ["gradle"]


def ensure_jpf_installed(
    env: dict[str, str],
    logger: JpfNotebookLogger,
    workspace_dir: Path,
    jpf_repo_url: str = "https://github.com/javapathfinder/jpf-core.git",
    jpf_branch: str = "java-17",
) -> JpfInstallStatus:
    workspace_dir.mkdir(parents=True, exist_ok=True)
    jpf_home = resolve_jpf_home(logger, workspace_dir)
    run_jpf_jar = jpf_home / "build" / "RunJPF.jar"
    site_properties = workspace_dir / "jpf_site.properties"

    status = JpfInstallStatus(
        jpf_home=jpf_home,
        run_jpf_jar=run_jpf_jar,
        site_properties=site_properties,
        install_command=["git", "clone", "--depth", "1", "--branch", jpf_branch, jpf_repo_url, str(jpf_home)],
        build_command=[*resolve_gradle_wrapper(jpf_home), "buildJars", "-x", "test"],
    )

    if not run_jpf_jar.exists():
        if jpf_home.exists():
            shutil.rmtree(jpf_home)
        install_result = run_shell_command(status.install_command, workspace_dir.parent, env, logger, "jpf_install")
        status.install_result = install_result
        status.install_success = install_result.exit_code == 0
        if not status.install_success:
            return status
    else:
        status.install_success = True

    if not run_jpf_jar.exists():
        build_result = run_shell_command(status.build_command, jpf_home, env, logger, "jpf_build")
        status.build_result = build_result
        status.build_success = build_result.exit_code == 0 and run_jpf_jar.exists()
    else:
        status.build_success = True

    site_properties.write_text(f"jpf-core = {jpf_path(jpf_home)}\n", encoding="utf-8")
    return status


def extract_class_name(java_path: Path) -> str:
    try:
        text = java_path.read_text(encoding="utf-8")
    except (OSError, UnicodeDecodeError):
        return java_path.stem
    match = CLASS_NAME_RE.search(text)
    return match.group(1) if match else java_path.stem


def has_main_method(java_path: Path) -> bool:
    try:
        text = java_path.read_text(encoding="utf-8")
    except (OSError, UnicodeDecodeError):
        return False
    return bool(MAIN_METHOD_RE.search(text))


def build_java_inventory(java_files: list[Path]) -> list[JavaClassInfo]:
    rows: list[JavaClassInfo] = []
    for path in java_files:
        package = extract_package_name(path)
        class_name = extract_class_name(path)
        qualified = f"{package}.{class_name}" if package else class_name
        rows.append(
            JavaClassInfo(
                file_path=path,
                package=package,
                class_name=class_name,
                qualified_name=qualified,
                has_main=has_main_method(path),
            )
        )
    return rows


def save_java_inventory(java_classes: list[JavaClassInfo], output_csv: Path) -> None:
    rows = [
        {
            "file_path": str(item.file_path),
            "package": item.package,
            "class_name": item.class_name,
        }
        for item in java_classes
    ]
    pd.DataFrame(rows).to_csv(output_csv, index=False)


def count_packages(java_classes: list[JavaClassInfo]) -> int:
    return len({item.package for item in java_classes if item.package})


def compute_repository_summary(
    repo_path: Path,
    java_classes: list[JavaClassInfo],
    build_tool: str,
    java_version: str,
) -> dict[str, Any]:
    total_size = sum(item.file_path.stat().st_size for item in java_classes if item.file_path.exists())
    return {
        "repository_name": repo_path.name,
        "repository_path": str(repo_path),
        "java_version": java_version.replace("\n", " | "),
        "build_tool": build_tool,
        "total_packages": count_packages(java_classes),
        "total_classes": len(java_classes),
        "repository_size_bytes": total_size,
    }


def discover_compiled_classpath_dirs(repo_path: Path) -> list[Path]:
    dirs: list[Path] = []
    markers = ("/target/classes", "/build/classes/java/main", "/build/classes", "/out/production")
    for path in repo_path.rglob("classes"):
        if path.name != "classes" or not path.is_dir():
            continue
        if ".git" in path.parts:
            continue
        normalized = str(path).replace("\\", "/").lower()
        if any(marker in normalized for marker in markers):
            dirs.append(path.resolve())
    return sorted(set(dirs))


def discover_sourcepath_dirs(repo_path: Path) -> list[Path]:
    dirs: list[Path] = []
    for marker in ("src/main/java", "src"):
        for path in repo_path.rglob(marker):
            if path.is_dir() and ".git" not in path.parts:
                if path.name == "java" or path.name == "src":
                    dirs.append(path.resolve())
    unique: list[Path] = []
    for path in sorted(set(dirs), key=lambda item: len(str(item)), reverse=True):
        if not any(str(path).startswith(str(existing)) for existing in unique):
            unique.append(path)
    return sorted(unique)


def classpath_string(classpath_dirs: list[Path]) -> str:
    return ";".join(jpf_path(path) for path in classpath_dirs)


def sourcepath_string(source_dirs: list[Path]) -> str:
    return ";".join(jpf_path(path) for path in source_dirs)


def write_project_jpf_properties(
    repo_path: Path,
    classpath_dirs: list[Path],
    source_dirs: list[Path],
    module_name: str = "sut",
) -> Path:
    properties_path = repo_path / "jpf.properties"
    content = "\n".join(
        [
            f"{module_name} = {jpf_path(repo_path)}",
            f"{module_name}.classpath = {classpath_string(classpath_dirs)}",
            f"{module_name}.sourcepath = {sourcepath_string(source_dirs)}",
            "listener.autoload = gov.nasa.jpf.listener.CoverageAnalyzer",
            "",
        ]
    )
    properties_path.write_text(content, encoding="utf-8")
    return properties_path


def write_jpf_application_file(
    output_dir: Path,
    qualified_name: str,
    classpath_dirs: list[Path],
    source_dirs: list[Path],
    target_args: str = "",
) -> Path:
    output_dir.mkdir(parents=True, exist_ok=True)
    safe_name = qualified_name.replace(".", "_")
    jpf_path_file = output_dir / f"{safe_name}.jpf"
    lines = [
        "@using = jpf-core",
        f"target = {qualified_name}",
    ]
    if target_args:
        lines.append(f"target.args = {target_args}")
    lines.extend(
        [
            f"classpath = {classpath_string(classpath_dirs)}",
            f"sourcepath = {sourcepath_string(source_dirs)}",
            "report.console.start = jpf,sut",
            "report.console.property_violation = error,trace",
            "search.multiple_errors = true",
            "",
        ]
    )
    jpf_path_file.write_text("\n".join(lines), encoding="utf-8")
    return jpf_path_file


def execute_compile_only(
    repo_path: Path,
    build_tool: str,
    env: dict[str, str],
    logger: JpfNotebookLogger,
) -> tuple[CommandResult, str]:
    if build_tool == "Maven":
        command = [*resolve_maven_command(repo_path, logger), "clean", "compile", "test-compile"]
    else:
        command = [*resolve_gradle_command(repo_path, logger), "clean", "classes", "testClasses"]
    result = run_shell_command(command, repo_path, env, logger, "build")
    raw = "\n".join(
        [
            f"===== {' '.join(command)} =====",
            combine_streams(result.stdout, result.stderr),
        ]
    )
    return result, raw


def run_jpf_for_class(
    install: JpfInstallStatus,
    jpf_file: Path,
    env: dict[str, str],
    logger: JpfNotebookLogger,
    qualified_name: str,
) -> JpfClassRun:
    command = [
        "java",
        "-jar",
        jpf_path(install.run_jpf_jar),
        f"+site={jpf_path(install.site_properties)}",
        jpf_path(jpf_file),
    ]
    started = time.perf_counter()
    completed = subprocess.run(
        command,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        check=False,
        env=env,
        cwd=str(jpf_file.parent),
    )
    elapsed = round(time.perf_counter() - started, 5)
    stdout = completed.stdout or ""
    stderr = completed.stderr or ""
    combined = combine_streams(stdout, stderr)
    success = "search finished" in combined.lower() and completed.returncode == 0
    metrics = parse_explicit_jpf_metrics(combined, qualified_name)
    if not success and completed.returncode != 0:
        logger.error(combined or f"JPF failed with exit code {completed.returncode}", step="jpf_run", class_name=qualified_name)
    return JpfClassRun(
        qualified_name=qualified_name,
        jpf_file=jpf_file,
        command=command,
        stdout=stdout,
        stderr=stderr,
        exit_code=completed.returncode,
        execution_time_seconds=elapsed,
        success=success,
        metrics=metrics,
    )


def execute_jpf_for_classes(
    install: JpfInstallStatus,
    java_classes: list[JavaClassInfo],
    classpath_dirs: list[Path],
    source_dirs: list[Path],
    env: dict[str, str],
    logger: JpfNotebookLogger,
    jpf_config_dir: Path,
) -> tuple[list[JpfClassRun], str]:
    chunks: list[str] = []
    runs: list[JpfClassRun] = []
    for item in java_classes:
        header = f"\n{'=' * 80}\nJPF RUN: {item.qualified_name}\n{'=' * 80}\n"
        chunks.append(header)
        if not item.has_main:
            message = f"Skipping JPF execution: {item.qualified_name} has no public static void main method."
            logger.error(message, step="jpf_config", class_name=item.qualified_name)
            chunks.append(message + "\n")
            continue
        if not classpath_dirs:
            message = f"Skipping JPF execution: no compiled classpath directories found for {item.qualified_name}."
            logger.error(message, step="jpf_config", class_name=item.qualified_name)
            chunks.append(message + "\n")
            continue

        target_args = "--skip-verify" if item.qualified_name.endswith("JacocoTrigger") else ""
        jpf_file = write_jpf_application_file(
            jpf_config_dir,
            item.qualified_name,
            classpath_dirs,
            source_dirs,
            target_args=target_args,
        )
        run = run_jpf_for_class(install, jpf_file, env, logger, item.qualified_name)
        logger.info(f"Executed JPF for {item.qualified_name} in {run.execution_time_seconds}s")
        runs.append(run)
        chunks.append(f"===== {' '.join(run.command)} =====\n")
        chunks.append(combine_streams(run.stdout, run.stderr))
        chunks.append("\n")
    return runs, "".join(chunks)


def parse_explicit_jpf_metrics(raw_output: str, source_class: str) -> list[dict[str, str]]:
    rows: list[dict[str, str]] = []
    stats_match = STATISTICS_BLOCK_RE.search(raw_output)
    stats_text = stats_match.group(1) if stats_match else raw_output

    def add(metric_name: str, metric_value: str, method: str = "") -> None:
        rows.append(
            {
                "metric_name": metric_name,
                "metric_value": metric_value,
                "source_class": source_class,
                "method": method,
            }
        )

    states_match = STATES_RE.search(stats_text)
    if states_match:
        add("Visited States", states_match.group(2), "statistics.states")
        add("New States", states_match.group(1), "statistics.states")
        add("Backtracked States", states_match.group(3), "statistics.states")
        add("End States", states_match.group(4), "statistics.states")

    depth_match = SEARCH_DEPTH_RE.search(stats_text)
    if depth_match:
        add("Search Depth", depth_match.group(1), "statistics.search")

    choice_match = CHOICE_GENERATORS_RE.search(stats_text)
    if choice_match:
        add("Choice Generators", choice_match.group(1).strip(), "statistics.choice_generators")

    transitions = TRANSITION_COUNT_RE.findall(raw_output)
    if transitions:
        add("Transition Count", str(len(transitions)), "trace.transition")
        add("Path Count", str(len(set(transitions))), "trace.transition")

    errors = ERROR_STATE_RE.findall(raw_output)
    if errors:
        add("Error States", str(len(errors)), "results.error")

    exceptions = EXCEPTION_RE.findall(raw_output)
    if exceptions:
        add("Exception States", str(len(set(exceptions))), "results.exception")

    if DEADLOCK_RE.search(raw_output):
        add("Deadlocks", "1", "results.deadlock")

    if "search started" in raw_output.lower():
        add("Execution Paths", str(len(TRACE_SECTION_RE.findall(raw_output)) or len(transitions)), "search")

    if stats_match:
        add("Search Statistics", stats_match.group(1).strip(), "statistics")

    elapsed_match = re.search(r"elapsed time:\s*(.+)", stats_text, re.IGNORECASE)
    if elapsed_match:
        add("Elapsed Time", elapsed_match.group(1).strip(), "statistics.elapsed_time")

    return rows


def extract_verbatim_sections(raw_console: str, output_dir: Path) -> dict[str, bool]:
    ensure_output_dir(output_dir)
    created: dict[str, bool] = {}

    traces = TRACE_SECTION_RE.findall(raw_console)
    if traces:
        path = output_dir / "error_trace.txt"
        path.write_text("\n".join(traces), encoding="utf-8")
        created["error_trace.txt"] = True

    transitions = [line for line in raw_console.splitlines() if "transition #" in line.lower()]
    if transitions:
        path = output_dir / "path_report.txt"
        path.write_text("\n".join(transitions) + "\n", encoding="utf-8")
        created["path_report.txt"] = True

    state_lines = [line for line in raw_console.splitlines() if line.strip().lower().startswith("states:")]
    if state_lines:
        path = output_dir / "visited_states.txt"
        path.write_text("\n".join(state_lines) + "\n", encoding="utf-8")
        created["visited_states.txt"] = True

    graph_lines = [
        line
        for line in raw_console.splitlines()
        if "choice generators" in line.lower() or "gov.nasa.jpf.vm.choice" in line
    ]
    if graph_lines:
        path = output_dir / "search_graph.txt"
        path.write_text("\n".join(graph_lines) + "\n", encoding="utf-8")
        created["search_graph.txt"] = True

    for name in JPF_OPTIONAL_ARTIFACTS:
        created.setdefault(name, (output_dir / name).exists())
    return created


def copy_generated_jpf_artifacts(search_dirs: list[Path], output_dir: Path) -> dict[str, bool]:
    copied: dict[str, bool] = {}
    for artifact in JPF_OPTIONAL_ARTIFACTS:
        destination = output_dir / artifact
        if destination.exists():
            copied[artifact] = True
            continue
        for directory in search_dirs:
            source = directory / artifact
            if source.exists():
                shutil.copy2(source, destination)
                copied[artifact] = True
                break
        copied.setdefault(artifact, destination.exists())
    return copied


def build_class_summary(runs: list[JpfClassRun]) -> pd.DataFrame:
    columns = [
        "Class",
        "Visited States",
        "Execution Paths",
        "Error States",
        "Exceptions",
        "Loops",
        "Search Depth",
        "Transitions",
        "JPF Success",
        "Raw Output Contains Statistics",
    ]
    if not runs:
        return pd.DataFrame(columns=columns)
    rows: list[dict[str, Any]] = []
    for run in runs:
        metric_map = {item["metric_name"]: item["metric_value"] for item in run.metrics}
        combined = combine_streams(run.stdout, run.stderr)
        rows.append(
            {
                "Class": run.qualified_name,
                "Visited States": metric_map.get("Visited States", ""),
                "Execution Paths": metric_map.get("Execution Paths", ""),
                "Error States": metric_map.get("Error States", ""),
                "Exceptions": metric_map.get("Exception States", ""),
                "Loops": metric_map.get("Backtracked States", ""),
                "Search Depth": metric_map.get("Search Depth", ""),
                "Transitions": metric_map.get("Transition Count", ""),
                "JPF Success": "Yes" if run.success else "No",
                "Raw Output Contains Statistics": "Yes" if "statistics" in combined.lower() else "No",
            }
        )
    return pd.DataFrame(rows)


def build_repository_metrics(
    java_classes: list[JavaClassInfo],
    runs: list[JpfClassRun],
    total_execution_time: float,
) -> pd.DataFrame:
    metrics_rows = [metric for run in runs for metric in run.metrics]
    metric_map: dict[str, list[str]] = {}
    for row in metrics_rows:
        metric_map.setdefault(row["metric_name"], []).append(row["metric_value"])

    def sum_numeric(name: str) -> int:
        total = 0
        for value in metric_map.get(name, []):
            try:
                total += int(value)
            except ValueError:
                continue
        return total

    summary = {
        "Total Classes": len(java_classes),
        "Total Methods": "",
        "Total Execution Paths": sum_numeric("Execution Paths"),
        "Visited States": sum_numeric("Visited States"),
        "Error States": sum_numeric("Error States"),
        "Exceptions": sum_numeric("Exception States"),
        "Loop Paths": sum_numeric("Backtracked States"),
        "Execution Time": total_execution_time,
        "Classes With Main": sum(1 for item in java_classes if item.has_main),
        "Classes Executed By JPF": len(runs),
    }
    return pd.DataFrame([summary])


def validate_path_metrics(raw_console: str, source_file: str = "jpf_console_output.txt") -> pd.DataFrame:
    rows: list[dict[str, str]] = []
    for metric in PATH_METRICS:
        evidence_parts: list[str] = []
        for label, pattern in METRIC_EVIDENCE_PATTERNS.get(metric, []):
            match = pattern.search(raw_console)
            if match:
                evidence_parts.append(f"{label}: {match.group(0)[:180]}")
        if evidence_parts:
            status = "Supported"
            evidence = " | ".join(evidence_parts)
            comments = "Explicit JPF output matched without inference."
        else:
            status = "No Evidence Found"
            evidence = ""
            comments = "No explicit JPF output matched this metric keyword/pattern."
        rows.append(
            {
                "Metric": metric,
                "Supported": status,
                "Evidence": evidence,
                "Source File": source_file,
                "Comments": comments,
            }
        )
    return pd.DataFrame(rows)


def preview_raw_output(raw_text: str, max_lines: int, source_path: Path) -> None:
    lines = raw_text.splitlines()
    print(f"Saved raw output: {source_path} ({len(lines)} lines)")
    preview = "\n".join(lines[:max_lines])
    print(preview)
    if len(lines) > max_lines:
        print(f"... truncated preview ({len(lines) - max_lines} additional lines in file)")


def build_dashboard_table(
    repo_stats: dict[str, Any],
    class_summary: pd.DataFrame,
    repository_metrics: pd.DataFrame,
) -> pd.DataFrame:
    repo_metric_row = repository_metrics.iloc[0].to_dict() if not repository_metrics.empty else {}
    depth_value = ""
    if not class_summary.empty and "Search Depth" in class_summary.columns:
        depth_values = class_summary["Search Depth"].replace("", pd.NA).dropna()
        if not depth_values.empty:
            depth_value = depth_values.iloc[0]
    return pd.DataFrame(
        [
            {
                "Metric": "Repository",
                "Value": repo_stats.get("repository_name", ""),
            },
            {
                "Metric": "Build Tool",
                "Value": repo_stats.get("build_tool", ""),
            },
            {
                "Metric": "Java Classes",
                "Value": repo_stats.get("total_classes", ""),
            },
            {
                "Metric": "Execution Paths",
                "Value": repo_metric_row.get("Total Execution Paths", ""),
            },
            {
                "Metric": "Visited States",
                "Value": repo_metric_row.get("Visited States", ""),
            },
            {
                "Metric": "Error States",
                "Value": repo_metric_row.get("Error States", ""),
            },
            {
                "Metric": "Exceptions",
                "Value": repo_metric_row.get("Exceptions", ""),
            },
            {
                "Metric": "Loop Paths",
                "Value": repo_metric_row.get("Loop Paths", ""),
            },
            {
                "Metric": "Search Depth",
                "Value": depth_value,
            },
        ]
    )



## Section 4 — Repository Information


In [ ]:
OUTPUT_PATH = Path(OUTPUT_DIR).resolve()
WORKSPACE_PATH = Path(WORKSPACE_DIR).resolve()
ERROR_LOG_PATH = OUTPUT_PATH / 'error_log.txt'

ensure_output_dir(OUTPUT_PATH)
logger = JpfNotebookLogger(ERROR_LOG_PATH)
JAVA_ENV = configure_java_runtime(logger)
JAVA_VERSION = java_version_text(JAVA_ENV)

REPO_PATH = resolve_repository_path(
    use_git_repo=USE_GIT_REPO,
    repo_url=REPO_URL,
    local_repo=LOCAL_REPO,
    workspace_dir=WORKSPACE_PATH,
    if_clone_exists=IF_CLONE_EXISTS,
    logger=logger,
    clone_depth=CLONE_DEPTH,
)

BUILD_TOOL = detect_build_tool(REPO_PATH)
JAVA_FILES = discover_java_files(REPO_PATH)
if not JAVA_FILES:
    raise FileNotFoundError('No Java source files found.')

JAVA_CLASSES = build_java_inventory(JAVA_FILES)
REPO_STATS = compute_repository_summary(REPO_PATH, JAVA_CLASSES, BUILD_TOOL, JAVA_VERSION)
REPOSITORY_SUMMARY_CSV = OUTPUT_PATH / 'repository_summary.csv'
pd.DataFrame([REPO_STATS]).to_csv(REPOSITORY_SUMMARY_CSV, index=False)

print(f"Repository Name: {REPO_STATS['repository_name']}")
print(f"Repository Path: {REPO_STATS['repository_path']}")
print(f"Java Version: {REPO_STATS['java_version']}")
print(f"Build Tool: {REPO_STATS['build_tool']}")
print(f"Total Packages: {REPO_STATS['total_packages']}")
print(f"Total Classes: {REPO_STATS['total_classes']}")
print(f"Repository Size (bytes): {REPO_STATS['repository_size_bytes']}")


## Section 5 — Discover Java Files


In [ ]:
INVENTORY_CSV = OUTPUT_PATH / 'java_files_inventory.csv'
save_java_inventory(JAVA_CLASSES, INVENTORY_CSV)
display(pd.read_csv(INVENTORY_CSV).head(20))


## Section 6 — Detect Build Tool


In [ ]:
print('Detected Build Tool:')
print(BUILD_TOOL)


## Section 7 — Build Repository


In [ ]:
PIPELINE_STARTED = time.perf_counter()
BUILD_RESULT, BUILD_CONSOLE_OUTPUT = execute_compile_only(
    REPO_PATH, BUILD_TOOL, JAVA_ENV, logger
)
print(f'Build success: {BUILD_RESULT.exit_code == 0}')
print(f'Build execution time (s): {BUILD_RESULT.execution_time_seconds}')


## Section 8 — Configure Java PathFinder


In [ ]:
JPF_INSTALL = ensure_jpf_installed(
    JAVA_ENV,
    logger,
    WORKSPACE_PATH,
    jpf_repo_url=JPF_REPO_URL,
    jpf_branch=JPF_BRANCH,
)
CLASSPATH_DIRS = discover_compiled_classpath_dirs(REPO_PATH)
SOURCEPATH_DIRS = discover_sourcepath_dirs(REPO_PATH)
PROJECT_JPF_PROPERTIES = write_project_jpf_properties(REPO_PATH, CLASSPATH_DIRS, SOURCEPATH_DIRS)
JPF_CONFIG_DIR = OUTPUT_PATH / 'jpf_configs'
JPF_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

print('JPF Home:', JPF_INSTALL.jpf_home)
print('RunJPF.jar:', JPF_INSTALL.run_jpf_jar)
print('site.properties:', JPF_INSTALL.site_properties)
print('Project jpf.properties:', PROJECT_JPF_PROPERTIES)
print('Classpath dirs:', [str(path) for path in CLASSPATH_DIRS])
print('Sourcepath dirs:', [str(path) for path in SOURCEPATH_DIRS])
print('\nGenerated jpf.properties:')
print(PROJECT_JPF_PROPERTIES.read_text(encoding='utf-8'))


## Section 9 — Execute Java PathFinder


In [ ]:
if not JPF_INSTALL.build_success:
    logger.error('JPF installation/build failed. Cannot execute JPF.', step='jpf_install')
    JPF_RUNS, JPF_CONSOLE_OUTPUT = [], BUILD_CONSOLE_OUTPUT
else:
    JPF_RUNS, JPF_CONSOLE_OUTPUT = execute_jpf_for_classes(
        JPF_INSTALL,
        JAVA_CLASSES,
        CLASSPATH_DIRS,
        SOURCEPATH_DIRS,
        JAVA_ENV,
        logger,
        JPF_CONFIG_DIR,
    )
    JPF_CONSOLE_OUTPUT = BUILD_CONSOLE_OUTPUT + '\n' + JPF_CONSOLE_OUTPUT
print(f'Classes executed by JPF: {len(JPF_RUNS)}')
print(f'Classes with main method: {sum(1 for item in JAVA_CLASSES if item.has_main)}')


## Section 10 — Raw Output Collection


In [ ]:
CONSOLE_PATH = OUTPUT_PATH / 'jpf_console_output.txt'
CONSOLE_PATH.write_text(JPF_CONSOLE_OUTPUT, encoding='utf-8')
SECTION_ARTIFACTS = extract_verbatim_sections(JPF_CONSOLE_OUTPUT, OUTPUT_PATH)
COPIED_ARTIFACTS = copy_generated_jpf_artifacts([JPF_CONFIG_DIR, REPO_PATH], OUTPUT_PATH)
print('Saved:', CONSOLE_PATH)
print('Section artifacts:', SECTION_ARTIFACTS)
print('Copied artifacts:', COPIED_ARTIFACTS)
preview_raw_output(JPF_CONSOLE_OUTPUT, RAW_OUTPUT_PREVIEW_LINES, CONSOLE_PATH)


## Section 11 — Extract JPF Results


In [ ]:
METRIC_ROWS = []
for run in JPF_RUNS:
    METRIC_ROWS.extend(run.metrics)
JPF_METRICS_DF = pd.DataFrame(METRIC_ROWS)
if JPF_METRICS_DF.empty:
    JPF_METRICS_DF = pd.DataFrame(columns=['metric_name', 'metric_value', 'source_class', 'method'])
JPF_METRICS_DF.to_csv(OUTPUT_PATH / 'jpf_metrics.csv', index=False)
display(JPF_METRICS_DF.head(30))


## Section 12 — Path Analysis


In [ ]:
PATH_VALIDATION_DF = validate_path_metrics(JPF_CONSOLE_OUTPUT, 'jpf_console_output.txt')
PATH_VALIDATION_DF.to_csv(OUTPUT_PATH / 'path_validation.csv', index=False)
display(PATH_VALIDATION_DF)


## Section 13 — Class Summary


In [ ]:
CLASS_SUMMARY_DF = build_class_summary(JPF_RUNS)
CLASS_SUMMARY_DF.to_csv(OUTPUT_PATH / 'class_summary.csv', index=False)
display(CLASS_SUMMARY_DF)


## Section 14 — Repository Summary


In [ ]:
TOTAL_EXECUTION_TIME = round(time.perf_counter() - PIPELINE_STARTED, 5)
REPOSITORY_METRICS_DF = build_repository_metrics(JAVA_CLASSES, JPF_RUNS, TOTAL_EXECUTION_TIME)
REPOSITORY_METRICS_DF.to_csv(OUTPUT_PATH / 'repository_metrics.csv', index=False)
display(REPOSITORY_METRICS_DF)


## Section 15 — Dashboard


In [ ]:
DASHBOARD_DF = build_dashboard_table(REPO_STATS, CLASS_SUMMARY_DF, REPOSITORY_METRICS_DF)
display(DASHBOARD_DF)


## Section 16 — Error Handling


In [ ]:
logger.write_errors()
if ERROR_LOG_PATH.exists():
    display(pd.read_csv(ERROR_LOG_PATH))
else:
    print('No errors logged.')
